<a href="https://colab.research.google.com/github/eelnayr/Animal-Classification/blob/main/Animal_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
from sklearn.utils import class_weight

In [ ]:
# convert all imag sizes to 64x64 picels and divide by 255 to normalize images sizes between [0,255]
# [0,255] helps model classify images to [0,1] for easy classification
def preprocess(image, label):

  # label: 0 for dog and 1 for cat
  return tf.image.resize(image, (64, 64)) / 255.0, label

# load and split data from tensorFlow libraries into train and test. Store as supervised data with image and its label
train_data, test_data = (tfds.load("cats_vs_dogs", split = ["train[:80%]", "train[80%:]"], as_supervised = True))

# preprocess train and test data into batches of 32 (of each label)
# put data in branches so model can learn faster by using parallel processing
# prefetch is data pipeline to streamline data being processes for next batch while creating current batch
train_data = train_data.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)
test_data = test_data.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)

In [ ]:
#covolutional neural network (CNN) is a neural network used for classification
#sequential is an API in keraas that stacks layers to build a CNN
model = tf.keras.Sequential((

    # the 16 filters (3x3) scan the input image, each detecting different features (ex: edges)
    # ReLU (Rectified Linear Unit) is an activation function that makes sure the layer focuses on important features
    # the model takes 64x64 pixel images with 3 color channnels (RGB)
    tf.keras.layers.Conv2D(16, (3, 3), activation = 'relu', input_shape = (64, 64, 3)),

    # takes feature map created by Conv2D and shrinks it by using a 2x2 matrix and retains important information from each matrix
    tf.keras.layers.MaxPooling2D((2,2)),

    # looks for patterns/features from previous layer
    tf.keras.layers.Conv2D(32, (3, 3), activation = 'relu'),

    # shrinks feature map amde by previous layer and retains important info
    tf.keras.layers.MaxPooling2D((2,2)),

    # averages values for every channel (RGB) in map
    tf.keras.layers.GlobalAveragePooling2D(),

    # classifier layer where 2 shows number of classes to identify (cat or dog)
    # softmax is an activation function that converts the scores generated by previous layer into probability between [0,1]
    tf.keras.layers.Dense(2, activation = 'softmax')
))

# configuring the model befoer training
# optimizer (adam): makes training faster and reduces data loss
# loss (spars_categorical_crossentropy): measures how far the model's predictions are from the actual values
# metrics (Accuracy): percentage of correct predictions made by the model during training
model.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

In [ ]:
# collects all labels from the training dataset
labels = np.concatenate([y for x, y in train_data], axis = 0)

# assigns each label weights based on frequency (lower frequency == higher weigth) so model focuses on all labels equally
class_weight_dict = dict(enumerate(class_weight.compute_class_weight('balanced', classes = np.unique(labels), y = labels)))

In [ ]:
# model will train on the train_data for 2 passes (epochs)
# validation_data = test_data: 20% test_data used to check if the model is learning not just memorizing (overfitting) images
# class_weight = class_weight_dict: model focuses on all classes equally
model.fit(train_data, validation_data = test_data, epochs = 2, class_weight = class_weight_dict)

In [ ]:
from tensorflow.keras.preprocessing import image

# load image of a cat or sog to test model
image_patch = "ENTER AN IMAGE HERE"

# resizes loaded image to 64x64 pixels
img = image.load_img(image_patch, target_size = (64, 64))

# converts pixels to numbers and normalizers those numbers to be between [0, 1]
img_array = np.expand_dims(image.img_to_array(img), axis = 0) / 255.0

# passes image into model
prediction = model.predict(img_array)

# returns the highest probability
label = np.argmax(prediction)

# returns cat or dog based on prediction
if label == 0:
  print("It's a Dog!")

else:
  print("It's a Cat!")